In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping



MAX_FEATURES = 10000
MAXLEN = 200
BATCH_SIZE = 32
EPOCHS = 15


def load_and_preprocess_data():
    print("Loading IMDb dataset...")

    (x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=MAX_FEATURES)

    print(f"Training samples: {len(x_train)}")
    print(f"Testing samples : {len(x_test)}")

    print("Padding sequences...")
    x_train = pad_sequences(x_train, maxlen=MAXLEN)
    x_test = pad_sequences(x_test, maxlen=MAXLEN)

    return x_train, y_train, x_test, y_test


def build_simple_rnn():
    model = Sequential([
        Embedding(MAX_FEATURES, 128, input_length=MAXLEN),
        SimpleRNN(64, dropout=0.2),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model


def build_lstm():
    model = Sequential([
        Embedding(MAX_FEATURES, 128, input_length=MAXLEN),
        Bidirectional(LSTM(64, dropout=0.2)),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model



def build_gru():
    model = Sequential([
        Embedding(MAX_FEATURES, 128, input_length=MAXLEN),
        Bidirectional(GRU(64, dropout=0.2)),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

def train_and_evaluate(model, x_train, y_train, x_test, y_test, model_name):
    print(f"\n--- Training {model_name} ---")

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )

    history = model.fit(
        x_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=1
    )

    test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)

    print(f"{model_name} - Test Loss: {test_loss:.4f}")
    print(f"{model_name} - Test Accuracy: {test_accuracy:.4f}")

    return history, test_accuracy


def analyze_predictions(model, x_test, y_test):
    predictions = (model.predict(x_test[:10]) > 0.5).astype("int32")

    word_index = imdb.get_word_index()
    reverse_word_index = {value: key for key, value in word_index.items()}

    def decode_review(text):
        return ' '.join([reverse_word_index.get(i - 3, "?") for i in text])

    print("\n--- Correct vs Wrong Predictions ---")

    for i in range(10):
        review = decode_review(x_test[i])

        actual = "Positive" if y_test[i] == 1 else "Negative"
        predicted = "Positive" if predictions[i][0] == 1 else "Negative"

        status = "CORRECT" if actual == predicted else "WRONG"

        print(f"\nReview: {review[:100]}...")
        print(f"Actual   : {actual}")
        print(f"Predicted: {predicted}")
        print(f"Status   : {status}")
        print("-" * 40)

def main():
    x_train, y_train, x_test, y_test = load_and_preprocess_data()

    models = {
        "Simple RNN": build_simple_rnn(),
        "LSTM": build_lstm(),
        "GRU": build_gru()
    }

    histories = {}
    accuracies = {}

    for name, model in models.items():
        history, accuracy = train_and_evaluate(
            model,
            x_train,
            y_train,
            x_test,
            y_test,
            name
        )

        histories[name] = history
        accuracies[name] = accuracy

    print("\n--- Model Comparison ---")

    for name, accuracy in accuracies.items():
        print(f"{name}: {accuracy:.4f}")

    best_model_name = max(accuracies, key=accuracies.get)
    best_model = models[best_model_name]

    print(f"\nBest Model: {best_model_name}")
    print("Reason: LSTM/GRU usually perform better than Simple RNN because they handle long-term dependencies better.")

    analyze_predictions(best_model, x_test, y_test)


if __name__ == "__main__":
    main()

Loading IMDb dataset...
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training samples: 25000
Testing samples : 25000
Padding sequences...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(



--- Training Simple RNN ---
Epoch 1/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 16s 18ms/step - accuracy: 0.5430 - loss: 0.6833 - val_accuracy: 0.6418 - val_loss: 0.6221
Epoch 2/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.6877 - loss: 0.5762 - val_accuracy: 0.7132 - val_loss: 0.5530
Epoch 3/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.7803 - loss: 0.4648 - val_accuracy: 0.7574 - val_loss: 0.5257
Epoch 4/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.7896 - loss: 0.4434 - val_accuracy: 0.6894 - val_loss: 0.6037
Epoch 5/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.8206 - loss: 0.4069 - val_accuracy: 0.7724 - val_loss: 0.5171
Epoch 6/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.8706 - loss: 0.3214 - val_accuracy: 0.7676 - val_loss: 0.5441
Epoch 7/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.8233 - loss: 0.4044 - val_accuracy: 0.7400 - val_loss: 0.6156
Epoch 8/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accur